# Генератор рукописных строк (EN + RU) — примеры

Шрифты в `assets/fonts_ru` и `assets/fonts_en` (`python scripts/fetch_fonts.py`)
или укажи свои папки.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from IPython.display import display
from src.synth import HandwrittenLineGenerator, make_generator

RU_FONTS = str(ROOT / 'assets' / 'fonts_ru')
EN_FONTS = str(ROOT / 'assets' / 'fonts_en')

## Сэмплим пары (текст, картинка)

Каждая строка — RU или EN (доля русских = `p_ru`). `*_text_dirs` пустые -> встроенные
словари; впиши свои папки с `.txt` для реального текста.

In [ ]:
gen = HandwrittenLineGenerator.from_dirs(
    ru_text_dirs=[], en_text_dirs=[],
    ru_font_dirs=RU_FONTS, en_font_dirs=EN_FONTS,
    p_ru=0.5, curriculum=False,
)
print('шрифтов: ru=%d en=%d' % (gen.fonts.n('ru'), gen.fonts.n('en')))

for i in range(6):
    img, text = gen.sample(make_generator(42, 0, i))
    print(text)
    display(img)

## Свои тексты (две папки) + длина + переносы

`len_chars=(мин,макс)` — длина (значит и размер картинки). `p_hyphenate` — доля переносов `-`
(только на реальном тексте). `p_words=p_random=0` -> только текст из корпуса.

In [ ]:
gen = HandwrittenLineGenerator.from_dirs(
    ru_text_dirs=['/data/ru_texts'],   # ваши русские .txt
    en_text_dirs=['/data/en_texts'],   # ваши английские .txt
    ru_font_dirs=RU_FONTS, en_font_dirs=EN_FONTS,
    p_ru=0.5, len_chars=(15, 45), p_hyphenate=0.3, curriculum=False,
)
print('файлов: ru=%d en=%d' % (len(gen.sampler._files['ru']), len(gen.sampler._files['en'])))
for i in range(6):
    img, text = gen.sample(make_generator(1, 0, i))
    print(text); display(img)

## В обучении (на лету)

Бесконечный поток -> `IterableDataset` с пер-воркерным сидом (см. `src/data.py`).

In [ ]:
import torch
from torch.utils.data import IterableDataset, DataLoader

class SynthLines(IterableDataset):
    def __init__(self, gen, base_seed=42):
        self.gen, self.base_seed = gen, base_seed
    def __iter__(self):
        info = torch.utils.data.get_worker_info(); wid = info.id if info else 0; i = 0
        while True:
            img, text = self.gen.sample(make_generator(self.base_seed, wid, i), step=i)
            yield {'image': img, 'text': text}; i += 1

loader = DataLoader(SynthLines(gen), batch_size=4, num_workers=0,
                    collate_fn=lambda b: ([x['image'] for x in b], [x['text'] for x in b]))
images, texts = next(iter(loader))
print(texts)
# дальше: processor(images=images).pixel_values + tokenizer(texts) -> TrOCR

Подсказки: `p_ru` — доля русских строк; `step` в `sample(rng, step)` управляет сложностью;
короткая сторона картинки = `output.min_side` (224), аспект натуральный; больше шрифтов —
в `assets/fonts_ru` / `assets/fonts_en`.